In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "created_house_price_prediction.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "updateabdullahi/house-price-prediction-dataset-clean-and-synthetic",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

C:\Users\HP\AppData\Local\Temp\ipykernel_12892\4206868113.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:      Area  Bedrooms  Bathrooms  Garage  YearBuilt Location        Price
0  8000.0       3.0        2.0     1.0     2004.0    Lagos  105402678.0
1  8100.0       2.0        2.0     1.0     2013.0   Kaduna   62530695.0
2  8200.0       2.0        1.0     0.0     2023.0     Kano   71488309.0
3  8300.0       3.0        1.0     0.0     2006.0     Kano   70608894.0
4  8400.0       3.0        2.0     1.0     2003.0    Lagos  107029977.0


In [3]:
print(df.columns)
print(df.shape)
# 51 label is NaN
print(df['Price'].isnull().sum())

Index(['Area', 'Bedrooms', 'Bathrooms', 'Garage', 'YearBuilt', 'Location',
       'Price'],
      dtype='str')
(2050, 7)
51


In [4]:
features = ['Area', 'Bedrooms', 'Bathrooms', 'Garage', 'YearBuilt', 'Location']
label = ['Price']

# drop before spliting the data
df = df.dropna()

X = df[features]
Y = df[label]
display(X.isnull().sum())
print(df.isnull().sum())

Area         0
Bedrooms     0
Bathrooms    0
Garage       0
YearBuilt    0
Location     0
dtype: int64

Area         0
Bedrooms     0
Bathrooms    0
Garage       0
YearBuilt    0
Location     0
Price        0
dtype: int64


In [5]:
# drop the null row
# X = X.dropna()
# Y = Y.dropna()
# print(X.isnull().sum())
# print(Y.isnull().sum())

In [6]:
print(Y)

            Price
0     105402678.0
1      62530695.0
2      71488309.0
3      70608894.0
4     107029977.0
...           ...
1994  200000000.0
1995  200000000.0
1996  200000000.0
1997  200000000.0
1998  200000000.0

[1999 rows x 1 columns]


# label is continuous value, so we are using Regression model

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
from sklearn.metrics import (
    r2_score,                          # How much variance the model explains
    explained_variance_score,           # Similar to R² but uses variance directly
    mean_squared_error,                 # Average of squared errors
    mean_absolute_error,                # Average of absolute errors
    median_absolute_error,              # Median of absolute errors (robust to outliers)
    mean_absolute_percentage_error,     # Average percentage error
    max_error                           # Worst single prediction error
)

def regression_report(y_true, y_pred, digits=4):
    r2 = r2_score(y_true, y_pred)
    explained_var = explained_variance_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    max_err = max_error(y_true, y_pred)
    metrics = {
        'Metric': [
            'R² Score',                      # Overall quality
            'Explained Variance',            # Variance explained
            'RMSE',                          # Error in original units
            'MSE',                           # Squared error (hard to interpret)
            'MAE',                           # Average error
            'Median AE',                     # Median error (robust)
            'MAPE (%)',                      # Percentage error
            'Max Error'                      # Worst case error
        ],
        'Value': [
            r2,                              # 0.8500
            explained_var,                   # 0.8498
            rmse,                            # 25000.00
            mse,                             # 625000000.00
            mae,                             # 18000.00
            median_ae,                       # 15000.00
            mape,                            # 4.2000
            max_err                          # 45000.00
        ]
    }
    report_df = pd.DataFrame(metrics)
    report_df['Value'] = report_df['Value'].round(digits)
    interpretations = {
        'R² Score': 'Higher is better (0-1) - Explained variance proportion',
        'Explained Variance': 'Higher is better (0-1) - Variance captured',
        'RMSE': 'Lower is better - Error in original units (dollars)',
        'MSE': 'Lower is better - Error in squared units (hard to interpret)',
        'MAE': 'Lower is better - Average absolute error in original units',
        'Median AE': 'Lower is better - Robust to outliers (median error)',
        'MAPE (%)': 'Lower is better - Average percentage error',
        'Max Error': 'Lower is better - Worst-case prediction error'
    }
    report_df['Interpretation'] = report_df['Metric'].map(interpretations)
    return report_df

In [9]:
print(X)

          Area  Bedrooms  Bathrooms  Garage  YearBuilt Location
0       8000.0       3.0        2.0     1.0     2004.0    Lagos
1       8100.0       2.0        2.0     1.0     2013.0   Kaduna
2       8200.0       2.0        1.0     0.0     2023.0     Kano
3       8300.0       3.0        1.0     0.0     2006.0     Kano
4       8400.0       3.0        2.0     1.0     2003.0    Lagos
...        ...       ...        ...     ...        ...      ...
1994  207400.0       3.0        2.0     0.0     2014.0   Jigawa
1995  207500.0       3.0        2.0     1.0     2011.0    Lagos
1996  207600.0       1.0        2.0     1.0     2004.0   Jigawa
1997  207700.0       1.0        1.0     1.0     2001.0    Lagos
1998  207800.0       3.0        1.0     1.0     2005.0    Lagos

[1999 rows x 6 columns]


In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 55)
print(X_test)
# print(Y_test)

          Area  Bedrooms  Bathrooms  Garage  YearBuilt Location
407    48700.0       5.0        3.0     0.0     2005.0   Kaduna
75     15500.0       4.0        2.0     1.0     2015.0    Lagos
238    31800.0       2.0        3.0     1.0     2007.0   Jigawa
1469  154900.0       4.0        1.0     1.0     2002.0    Lagos
1592  167200.0       1.0        2.0     0.0     2022.0     Kano
...        ...       ...        ...     ...        ...      ...
1890  197000.0       5.0        1.0     0.0     1996.0   Jigawa
1113  119300.0       5.0        2.0     0.0     1996.0    Abuja
804    88400.0       4.0        2.0     1.0     2005.0   Jigawa
688    76800.0       3.0        2.0     1.0     2006.0     Kano
932   101200.0       1.0        1.0     1.0     2023.0    Abuja

[400 rows x 6 columns]


In [11]:
# Location is string, encode it so that it help training the model
'''
# doing this is wrong: linear will take it as number for calculation
label_encoder = LabelEncoder()
X_train['Location'] = label_encoder.fit_transform(X_train['Location'])
X_test['Location'] = label_encoder.fit_transform(X_test['Location'])
print(X_test)
'''

# drop_first=True prevents the dummy variable trap (multicollinearity)
X = pd.get_dummies(X, columns=['Location'], drop_first=True)

from category_encoders import TargetEncoder
encoder = TargetEncoder(cols=['Location'])
X_train['Location'] = encoder.fit_transform(X_train['Location'], Y_train)
X_test['Location'] = encoder.transform(X_test['Location'])

In [12]:
print(X_test)

          Area  Bedrooms  Bathrooms  Garage  YearBuilt      Location
407    48700.0       5.0        3.0     0.0     2005.0  1.919576e+08
75     15500.0       4.0        2.0     1.0     2015.0  1.947749e+08
238    31800.0       2.0        3.0     1.0     2007.0  1.882244e+08
1469  154900.0       4.0        1.0     1.0     2002.0  1.947749e+08
1592  167200.0       1.0        2.0     0.0     2022.0  1.893524e+08
...        ...       ...        ...     ...        ...           ...
1890  197000.0       5.0        1.0     0.0     1996.0  1.882244e+08
1113  119300.0       5.0        2.0     0.0     1996.0  1.945900e+08
804    88400.0       4.0        2.0     1.0     2005.0  1.882244e+08
688    76800.0       3.0        2.0     1.0     2006.0  1.893524e+08
932   101200.0       1.0        1.0     1.0     2023.0  1.945900e+08

[400 rows x 6 columns]


# using LinearRegression

In [13]:
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()
linear_model.fit(X_train, Y_train)

prediction = linear_model.predict(X_test)
regression_report(Y_test, prediction)

,Metric,Value,Interpretation
0,R² Score,1.981000e-01,Higher is better (0-1) - Explained variance pr...
1,Explained Variance,2.027000e-01,Higher is better (0-1) - Variance captured
2,RMSE,1.968198e+07,Lower is better - Error in original units (dol...
3,MSE,3.873802e+14,Lower is better - Error in squared units (hard...
4,MAE,1.327498e+07,Lower is better - Average absolute error in or...
5,Median AE,1.011732e+07,Lower is better - Robust to outliers (median e...
6,MAPE (%),8.510000e+00,Lower is better - Average percentage error
7,Max Error,9.667555e+07,Lower is better - Worst-case prediction error


# using LassoRegression

In [14]:
from sklearn.linear_model import Lasso
lasso_model = Lasso()
lasso_model.fit(X_train, Y_train)

prediction = lasso_model.predict(X_test)

regression_report(Y_test, prediction)

,Metric,Value,Interpretation
0,R² Score,1.966000e-01,Higher is better (0-1) - Explained variance pr...
1,Explained Variance,2.013000e-01,Higher is better (0-1) - Variance captured
2,RMSE,1.969972e+07,Lower is better - Error in original units (dol...
3,MSE,3.880790e+14,Lower is better - Error in squared units (hard...
4,MAE,1.330582e+07,Lower is better - Average absolute error in or...
5,Median AE,9.928060e+06,Lower is better - Robust to outliers (median e...
6,MAPE (%),8.523200e+00,Lower is better - Average percentage error
7,Max Error,9.676399e+07,Lower is better - Worst-case prediction error


In [16]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.fit_transform(X_test)
print(X_test)

[[-1.08317706  1.36208335  1.51752542 -1.         -0.47617162  0.00714742]
 [-1.67241729  0.65725367  0.23960928  1.          0.69163125  1.05592661]
 [-1.38312164 -0.75240568  1.51752542  1.         -0.24261105 -1.38258736]
 ...
 [-0.37857354  0.65725367  0.23960928  1.         -0.47617162 -1.38258736]
 [-0.58445265 -0.047576    0.23960928  1.         -0.35939134 -0.96266176]
 [-0.15139658 -1.45723535 -1.03830686  1.          1.62587355  0.98708   ]]


In [17]:
# after transform: 🥲worse
lasso_model = Lasso()
lasso_model.fit(X_train, Y_train)

prediction = lasso_model.predict(X_test)

regression_report(Y_test, prediction)

,Metric,Value,Interpretation
0,R² Score,1.867000e-01,Higher is better (0-1) - Explained variance pr...
1,Explained Variance,1.974000e-01,Higher is better (0-1) - Variance captured
2,RMSE,1.982056e+07,Lower is better - Error in original units (dol...
3,MSE,3.928544e+14,Lower is better - Error in squared units (hard...
4,MAE,1.367040e+07,Lower is better - Average absolute error in or...
5,Median AE,1.055206e+07,Lower is better - Robust to outliers (median e...
6,MAPE (%),8.669000e+00,Lower is better - Average percentage error
7,Max Error,9.531891e+07,Lower is better - Worst-case prediction error


In [19]:
from sklearn.linear_model import Ridge
Ridge_model = Ridge(alpha = 1.0)
Ridge_model.fit(X_train, Y_train)

prediction = Ridge_model.predict(X_test)

regression_report(Y_test, prediction)

,Metric,Value,Interpretation
0,R² Score,1.868000e-01,Higher is better (0-1) - Explained variance pr...
1,Explained Variance,1.975000e-01,Higher is better (0-1) - Variance captured
2,RMSE,1.981937e+07,Lower is better - Error in original units (dol...
3,MSE,3.928073e+14,Lower is better - Error in squared units (hard...
4,MAE,1.366621e+07,Lower is better - Average absolute error in or...
5,Median AE,1.055085e+07,Lower is better - Robust to outliers (median e...
6,MAPE (%),8.667200e+00,Lower is better - Average percentage error
7,Max Error,9.533444e+07,Lower is better - Worst-case prediction error


In [21]:
print(X_train.shape)

(1599, 6)


# using HistGradientBoostingRegressor
> this is from AI, the efficient is really high
>

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Clean missing values and define features/target
df_clean = df.dropna().copy()
features = ['Area', 'Bedrooms', 'Bathrooms', 'Garage', 'YearBuilt', 'Location']
X = df_clean[features]
Y = df_clean['Price']

# 2. Split data before applying transformations
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=55)

# 3. One-hot encode categorical features safely
X_train = pd.get_dummies(X_train, columns=['Location'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Location'], drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 4. Fit a tree-based regressor (handles non-linear capping without feature scaling)
model = HistGradientBoostingRegressor(random_state=55)
model.fit(X_train, Y_train)

# 5. Evaluate predictions
predictions = model.predict(X_test)
regression_report(Y_test, predictions)

,Metric,Value,Interpretation
0,R² Score,9.930000e-01,Higher is better (0-1) - Explained variance pr...
1,Explained Variance,9.930000e-01,Higher is better (0-1) - Variance captured
2,RMSE,1.836674e+06,Lower is better - Error in original units (dol...
3,MSE,3.373371e+12,Lower is better - Error in squared units (hard...
4,MAE,4.802401e+05,Lower is better - Average absolute error in or...
5,Median AE,3.760513e+03,Lower is better - Robust to outliers (median e...
6,MAPE (%),3.651000e-01,Lower is better - Average percentage error
7,Max Error,1.906055e+07,Lower is better - Worst-case prediction error


> I seem to not understand how to split and know about the data yet
>
> how clean the data isn't correct too
>
> this is the most important part which makes the model under performed, beside the model can't is most fit for this dataset